## Question 1: Sequential Task Pipeline
Task:
Implement a 4-step document workflow using Gemini-powered ConversableAgents.
The steps are:

DraftWriter → Writes a short article draft on a given topic.

EditorAgent → Refines the grammar, structure, and tone of the draft.

PlagiarismChecker → Checks if the content is original or copied.

PublisherAgent → Finalizes the article and prepares it for publishing.

👉 Each agent should take the previous agent's output and continue the task without losing context.
👉 Print each step’s output clearly.
👉 Use generate_reply() for all interactions.

In [6]:
from autogen import ConversableAgent

# Config for Gemini
config = {
    "config_list": [{
        "model": "models/gemini-1.5-pro-latest",
        "api_key": "slUXIYX23q03Lf0ttYkiCtCeY2z4u7gT",
        "api_type": "google"
    }]
}

# Define agents
draft_writer = ConversableAgent(name="DraftWriter", llm_config=config)
editor = ConversableAgent(name="EditorAgent", llm_config=config)
plagiarism_checker = ConversableAgent(name="PlagiarismChecker", llm_config=config)
publisher = ConversableAgent(name="PublisherAgent", llm_config=config)

# Step 1: DraftWriter creates a draft
draft = draft_writer.generate_reply(messages=[
    {"role": "user", "content": "Write a short article about the importance of clean drinking water."}
])

# Step 2: EditorAgent refines the draft
edited = editor.generate_reply(messages=[
    {"role": "user", "content": draft["content"]}
])

# Step 3: PlagiarismChecker checks for originality
plagiarism_check = plagiarism_checker.generate_reply(messages=[
    {"role": "user", "content": f"Check for plagiarism in the following text:\n\n{edited['content']}"}
])

# Step 4: PublisherAgent finalizes it for publishing
published = publisher.generate_reply(messages=[
    {"role": "user", "content": edited["content"]}
])

# Print results
print("Draft:\n", draft["content"], "\n")
print("Edited:\n", edited["content"], "\n")
print("Plagiarism Check:\n", plagiarism_check["content"], "\n")
print("Published:\n", published["content"], "\n")


[autogen.oai.client: 01-31 16:42:21] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 01-31 16:42:21] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 01-31 16:42:22] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 01-31 16:42:22] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.

>>>>>>>> USING AUTO REPLY...


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: slUXIYX2********************u7gT. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

## Question 2. Task Decomposer (Planner + Workers)
Question: Use a PlannerAgent to break down the goal: "Create a business plan for a startup in mental health tech".
Then use MarketAgent, TechAgent, and FinancialAgent to address each part.

In [7]:
from autogen import ConversableAgent
import textwrap

# Config for Gemini
config = {
    "config_list": [{
        "model": "models/gemini-1.5-pro-latest",
        "api_key": "slUXIYX23q03Lf0ttYkiCtCeY2z4u7gT",
        "api_type": "google"
    }]
}

# Define agents
planner = ConversableAgent(name="PlannerAgent", llm_config=config)
market_agent = ConversableAgent(name="MarketAgent", llm_config=config)
tech_agent = ConversableAgent(name="TechAgent", llm_config=config)
finance_agent = ConversableAgent(name="FinancialAgent", llm_config=config)

# Utility function to trim long text
def truncate_text(text, max_words=200):
    words = text.split()
    return ' '.join(words[:max_words]) if len(words) > max_words else text

# Step 1: PlannerAgent breaks down the goal
planning = planner.generate_reply(messages=[
    {"role": "user", "content": "Break down the goal: Create a business plan for a startup in mental health tech"}
])

plan_text = planning["content"]
print("📋 Planner Breakdown:\n", plan_text, "\n")

# Truncate if too long
short_plan = truncate_text(plan_text)

# Step 2: MarketAgent handles the market analysis
market = market_agent.generate_reply(messages=[
    {"role": "user", "content": f"From this plan, focus only on market analysis:\n\n{short_plan}"}
])

# Step 3: TechAgent handles the technology plan
tech = tech_agent.generate_reply(messages=[
    {"role": "user", "content": f"From this plan, focus only on technology implementation:\n\n{short_plan}"}
])

# Step 4: FinancialAgent handles the financial strategy
finance = finance_agent.generate_reply(messages=[
    {"role": "user", "content": f"From this plan, focus only on financial planning:\n\n{short_plan}"}
])

# Print Outputs
print("📊 Market Analysis:\n", market["content"], "\n")
print("💻 Tech Strategy:\n", tech["content"], "\n")
print("💰 Financial Plan:\n", finance["content"], "\n")


[autogen.oai.client: 01-31 16:42:31] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 01-31 16:42:31] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 01-31 16:42:31] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 01-31 16:42:31] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.

>>>>>>>> USING AUTO REPLY...


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: slUXIYX2********************u7gT. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

## Question 3. Role-Based Chat (Specialized Roles)
Question: Assign roles to agents:

DoctorAgent gives medical insight

DietitianAgent suggests a meal plan

FitnessCoachAgent suggests workouts
For a given user query like: “Help me manage diabetes naturally.”

In [8]:
from autogen import ConversableAgent

# Gemini API config
config = {
    "config_list": [{
        "model": "models/gemini-1.5-pro-latest",
        "api_key": "WzDtVfOxFhdWSN2bxD9DVrosjHesRyKe",
        "api_type": "google"
    }]
}

# Define role-based agents
doctor = ConversableAgent(name="DoctorAgent", llm_config=config)
dietitian = ConversableAgent(name="DietitianAgent", llm_config=config)
fitness_coach = ConversableAgent(name="FitnessCoachAgent", llm_config=config)

# Shared user query
user_query = "Help me manage diabetes naturally."

# DoctorAgent gives medical insights
medical_reply = doctor.generate_reply(messages=[
    {"role": "user", "content": f"You are a medical professional. Provide insights for: {user_query}"}
])

# DietitianAgent suggests a meal plan
diet_reply = dietitian.generate_reply(messages=[
    {"role": "user", "content": f"You are a certified dietitian. Suggest a diabetes-friendly meal plan for: {user_query}"}
])

# FitnessCoachAgent suggests workouts
fitness_reply = fitness_coach.generate_reply(messages=[
    {"role": "user", "content": f"You are a fitness coach. Suggest suitable workouts for someone who asked: {user_query}"}
])

# Print each agent's specialized output
print("🩺 DoctorAgent (Medical Insight):\n", medical_reply["content"], "\n")
print("🥗 DietitianAgent (Meal Plan):\n", diet_reply["content"], "\n")
print("🏋️ FitnessCoachAgent (Workout Suggestions):\n", fitness_reply["content"], "\n")


[autogen.oai.client: 01-31 16:42:48] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 01-31 16:42:48] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 01-31 16:42:48] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.

>>>>>>>> USING AUTO REPLY...


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: WzDtVfOx********************RyKe. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

## Question 4. Multi-Language Translator Agents
Question: Input a paragraph in English.
EnglishToSpanish, SpanishToFrench, and FrenchToGerman agents must translate it in chain.
Output should be the final German translation.

In [9]:
from autogen import ConversableAgent

# Gemini API Configuration
config = {
    "config_list": [{
        "model": "models/gemini-1.5-pro-latest",
        "api_key": "WzDtVfOxFhdWSN2bxD9DVrosjHesRyKe",
        "api_type": "google"
    }]
}

# Create translator agents
eng_to_spanish = ConversableAgent(name="EnglishToSpanish", llm_config=config)
spanish_to_french = ConversableAgent(name="SpanishToFrench", llm_config=config)
french_to_german = ConversableAgent(name="FrenchToGerman", llm_config=config)

# Input paragraph
paragraph = "Nature is the best medicine. A walk in the forest or time spent near water can heal the soul and calm the mind."

# Step 1: English ➡️ Spanish
spanish_reply = eng_to_spanish.generate_reply(messages=[
    {"role": "user", "content": f"Translate this to Spanish: {paragraph}"}
])

# Step 2: Spanish ➡️ French
french_reply = spanish_to_french.generate_reply(messages=[
    {"role": "user", "content": f"Translate this to French: {spanish_reply['content']}"}
])

# Step 3: French ➡️ German
german_reply = french_to_german.generate_reply(messages=[
    {"role": "user", "content": f"Translate this to German: {french_reply['content']}"}
])

# Final Output
print("🧾 Final German Translation:\n", german_reply["content"])


[autogen.oai.client: 01-31 16:42:57] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 01-31 16:42:57] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 01-31 16:42:57] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.

>>>>>>>> USING AUTO REPLY...


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: WzDtVfOx********************RyKe. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

## Question 5: Fact Checker Agent Flow (Truth vs. Fake Detection)

🎯 Task: Create a pipeline to verify a social media claim using 3 agents:

ClaimExtractorAgent: Extracts main claim from user input

FactCheckerAgent: Determines if it's real or fake

ReporterAgent: Outputs a news-style response

In [10]:
from autogen import ConversableAgent

config = {
    "config_list": [{
        "model": "models/gemini-1.5-pro-latest",
        "api_key": "qEtBadmtLM2AxLuwDbJxtfqlHDpRrs2B",  # Replace with actual key
        "api_type": "google"
    }]
}

# Agents
extractor = ConversableAgent(name="ClaimExtractorAgent", llm_config=config)
checker = ConversableAgent(name="FactCheckerAgent", llm_config=config)
reporter = ConversableAgent(name="ReporterAgent", llm_config=config)

# User social media input
social_post = "I saw a post saying drinking hot water every 30 minutes kills COVID-19."

# Step 1: Extract the main claim
step1 = extractor.generate_reply(messages=[{"role": "user", "content": social_post}])

# Step 2: Check the factual accuracy
step2 = checker.generate_reply(messages=[{"role": "user", "content": step1["content"]}])

# Step 3: Generate a news-style report
step3 = reporter.generate_reply(messages=[{"role": "user", "content": step2["content"]}])

print("📌 Extracted Claim:\n", step1["content"])
print("\n🔍 Fact Check Result:\n", step2["content"])
print("\n📰 Final Report:\n", step3["content"])


[autogen.oai.client: 01-31 16:43:12] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 01-31 16:43:12] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.
[autogen.oai.client: 01-31 16:43:12] {124} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.

>>>>>>>> USING AUTO REPLY...


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: qEtBadmt********************rs2B. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}